# ساخت checkpointهای تمام‌لایهٔ Qwen MoE (Top-2 / 32 experts)

این Notebook برای Google Colab آماده شده است. خودش خودکار اجرا نمی‌شود؛ Runtime را به GPU رایگانِ در دسترس تنظیم کنید و سلول‌ها را به‌ترتیب اجرا کنید. هیچ سرویس پولی فعال نمی‌شود.

خروجی این مرحله checkpoint جداگانه برای FFN هر لایه است، نه هنوز یک مدل کامل قابل بارگذاری در Transformers. فایل‌ها در Google Drive ذخیره می‌شوند تا با قطع اتصال Colab باقی بمانند. اجرای ۲۴ لایه ممکن است زمان‌بر باشد.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK = Path('/content/drive/MyDrive/qwen-moe-full-build')
WORK.mkdir(parents=True, exist_ok=True)
print('Output folder:', WORK)

In [ ]:
!git clone https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git /content/Qwen-MoE-0.4B-
%cd /content/Qwen-MoE-0.4B-
!pip -q install -U transformers accelerate safetensors datasets

In [ ]:
import torch, os, json, subprocess
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('WARNING: no GPU detected. Full 24-layer run on CPU may take much longer.')

## گردآوری دادهٔ فارسی واقعی

این سلول ۲۰۰ سند فارسی واقعی از منابع مشخص‌شده در اسکریپت پروژه می‌گیرد. اگر منابع یا شبکه در دسترس نباشند، کار متوقف می‌شود و دادهٔ ساختگی جایگزین نمی‌شود. این حجم برای شروع و بررسی مسیر فنی است، نه آموزش نهایی.

In [ ]:
!python scripts/collect_persian_corpus.py --count 200 --min-chars 160 --max-chars 3000 --max-scan 30000 --output /content/fa_corpus.txt --manifest-output /content/fa_corpus.manifest.json
import shutil
shutil.copy2('/content/fa_corpus.txt', WORK / 'fa_corpus.txt')
shutil.copy2('/content/fa_corpus.manifest.json', WORK / 'fa_corpus.manifest.json')
print('Saved corpus and source/license manifest to Drive.')

## آموزش اصلاح‌شده: ابتدا پایلوت لایهٔ صفر

روش قبلی تقسیم تصادفی توکن‌ها داشت و Router می‌توانست بار را روی چند متخصص متمرکز کند. نسخهٔ فعلی دادهٔ ارزیابی را بر اساس سند جدا می‌کند، خطای نسبی را بهینه می‌کند و loss توازن بار Router را اضافه می‌کند.

**مهم:** خروجی در `checkpoints_v2` ذخیره می‌شود تا checkpointهای قبلی دست‌نخورده بمانند. ابتدا فقط لایهٔ صفر را اجرا و معیارهای ارزیابی و کیفیت پاسخ را بررسی کنید. سلول‌های لایه‌های دیگر را فقط پس از تأیید پایلوت اجرا کنید. ۲۰۰ سند هنوز دادهٔ محدود برای آموزش نهایی است.

In [ ]:
# Layer range 0-0
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "{WORK}/checkpoints_v2" --num-experts 32 --top-k 2 --epochs 10 --max-length 256 --batch-size 1 --train-batch-size 256 --balance-loss-weight 0.01 --eval-fraction 0.1 --device auto --start-layer 0 --end-layer 0
print('Finished range 0-0. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
pilot = torch.load(WORK / 'checkpoints_v2' / 'layer_00_top2_32experts.pt', map_location='cpu', weights_only=False)
print('Layer 0 pilot metrics')
for key in ('tokens_captured', 'train_documents', 'eval_documents', 'eval_mse', 'eval_relative_error', 'eval_cosine_similarity', 'balance_loss_weight'):
    print(f'{key}: {pilot.get(key)}')
print('Checkpoint:', WORK / 'checkpoints_v2' / 'layer_00_top2_32experts.pt')
print('Do not start the remaining layer ranges until the held-out metrics and generation tests have been reviewed.')

In [ ]:
# Layer range 1-4
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "{WORK}/checkpoints_v2" --num-experts 32 --top-k 2 --epochs 10 --max-length 256 --batch-size 1 --train-batch-size 256 --balance-loss-weight 0.01 --eval-fraction 0.1 --device auto --start-layer 1 --end-layer 4
print('Finished range 1-4. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 5-8
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "{WORK}/checkpoints_v2" --num-experts 32 --top-k 2 --epochs 10 --max-length 256 --batch-size 1 --train-batch-size 256 --balance-loss-weight 0.01 --eval-fraction 0.1 --device auto --start-layer 5 --end-layer 8
print('Finished range 5-8. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 9-12
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "{WORK}/checkpoints_v2" --num-experts 32 --top-k 2 --epochs 10 --max-length 256 --batch-size 1 --train-batch-size 256 --balance-loss-weight 0.01 --eval-fraction 0.1 --device auto --start-layer 9 --end-layer 12
print('Finished range 9-12. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 13-16
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "{WORK}/checkpoints_v2" --num-experts 32 --top-k 2 --epochs 10 --max-length 256 --batch-size 1 --train-batch-size 256 --balance-loss-weight 0.01 --eval-fraction 0.1 --device auto --start-layer 13 --end-layer 16
print('Finished range 13-16. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 17-20
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "{WORK}/checkpoints_v2" --num-experts 32 --top-k 2 --epochs 10 --max-length 256 --batch-size 1 --train-batch-size 256 --balance-loss-weight 0.01 --eval-fraction 0.1 --device auto --start-layer 17 --end-layer 20
print('Finished range 17-20. Verify checkpoint files in Google Drive before continuing.')

## بررسی فایل‌های تولیدشده

پس از اجرای همهٔ بازه‌ها، باید ۲۴ checkpoint داشته باشید. اگر تعداد کمتر است، بازه‌های مربوط به لایه‌های باقی‌مانده را اجرا کنید. این فایل‌ها هنوز مدل نهایی قابل بارگذاری یا GGUF نیستند.

In [ ]:
ckpt_dir = WORK / 'checkpoints_v2'
files = sorted(ckpt_dir.glob('layer_*_top2_32experts.pt'))
print(f'Found {len(files)} / 24 layer checkpoints')
for f in files: print(f.name, f.stat().st_size, 'bytes')
missing = [i for i in range(24) if not (ckpt_dir / f'layer_{i:02d}_top2_32experts.pt').exists()]
print('Missing layers:', missing)
assert len(files) == 24, 'Not all layer checkpoints exist yet; run missing ranges first.'

## بعد از این مرحله

باید کلاس مدل سازگار با Transformers پیاده‌سازی شود تا همهٔ checkpointها را بارگذاری کند؛ سپس ذخیره/بارگذاری مجدد، inference کامل، آزمون‌های فارسی و استدلال، و بعد بررسی کوانتیزه‌سازی و runtime اندروید انجام می‌شود. این Notebook ادعا نمی‌کند که مدل کامل یا کیفیت/سرعت نهایی را ساخته یا تأیید کرده است.